# Week1 복습과제 - ResNet

- 교재: 딥러닝 파이토치 교과서 6.1.5 ResNet (p.312~340)
- 범위 안의 코드 6-65 ~ 6-89를 교재에 나온 그대로 옮겨 적고 실행함
- 데이터만 길벗 공식 저장소에서 내려받아 교재와 같은 경로에 두고 씀

## ResNet 정리

- 신경망을 깊게 쌓으면 어느 순간부터 성능이 오히려 나빠짐. 논문에서도 56층이 20층보다 훈련 오차·테스트 오차가 모두 컸음
- 이를 해결하려고 레지듀얼 블록을 도입함. 블록을 지난 결과에 입력 x를 그대로 더해 주는 숏컷(스킵 연결)을 만듦
- 아이덴티티 매핑은 입력 x가 어떤 함수를 통과하더라도 다시 x라는 형태로 출력되도록 하는 것임. 식으로는 H(x) = F(x) + x 가 됨
- 숏컷 덕분에 기울기가 전달되는 경로가 생겨서 층이 152개까지 깊어져도 학습이 됨
- 계층을 계속 쌓으면 파라미터가 늘어나므로 병목 블록을 둠. ResNet18·34는 기본 블록(3x3 두 개), ResNet50·101·152는 병목 블록(1x1, 3x3, 1x1)을 씀
- 병목 블록은 3x3 앞뒤에 1x1 합성곱을 붙여 채널을 줄였다가 다시 늘림. 그래서 더 깊은데도 파라미터가 39.3216M에서 6.9632M으로 줄어듦
- 입력과 출력의 형태가 다르면 더할 수 없으므로 다운샘플이 필요함. 스트라이드 2를 가진 1x1 합성곱 한 층을 숏컷에 연결해 형태를 맞춤
- 입출력 차원이 같은 것을 아이덴티티 블록, 차원을 맞춰 줘야 하는 것을 프로젝션 숏컷(합성곱 블록)이라고 함
- 정리하면 ResNet은 VGG19 구조를 뼈대로 삼고 합성곱층을 더 쌓은 뒤 숏컷을 추가한 것임

## 데이터 준비

- 교재는 `../chap06/data/dogs-vs-cats/` 아래에 Cat, Dog 폴더가 있다고 보고 코드를 씀
- 코랩에는 그 폴더가 없으므로 길벗 공식 저장소의 dogs-vs-cats.zip을 받아 같은 경로를 만들어 둠
- 아래 셀은 경로를 맞추기 위한 준비 셀이고, 이 다음부터는 교재 코드를 그대로 씀

In [ ]:
import os
import urllib.request
import zipfile

os.makedirs('/content/chap06/data', exist_ok=True)

url = 'https://raw.githubusercontent.com/gilbutITbook/080289/main/chap06/data/dogs-vs-cats.zip'
zip_path = '/content/chap06/data/dogs-vs-cats.zip'

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/chap06/data/dogs-vs-cats')

os.makedirs('/content/notebook', exist_ok=True)
os.chdir('/content/notebook')

print(os.listdir('../chap06/data/dogs-vs-cats'))

### 코드 6-65 필요한 라이브러리 호출

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models

import matplotlib.pyplot as plt
import numpy as np

import copy
from collections import namedtuple
import os
import random
import time

import cv2
from torch.utils.data import DataLoader, Dataset
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

- 네임드튜플은 파이썬 자료형 중 하나임. 튜플의 성질을 갖고 있지만 인덱스뿐만 아니라 키 값으로도 데이터에 접근할 수 있음

In [ ]:
from collections import namedtuple

Student = namedtuple('Student', ['name','age','DOB'])
S = Student('홍길동', '19', '187')

print("The Student age using index is : ", end="")
print(S[1])

print("The Student name using keyname is : ", end="")
print(S.name)

### 코드 6-66 이미지 데이터 전처리

- 훈련 데이터에는 무작위 자르기와 좌우 반전을 적용함
- 검증과 테스트 데이터에는 256으로 크기를 맞춘 뒤 가운데를 잘라 씀

In [ ]:
class ImageTransform():
    def __init__(self, resize, mean, std):
        self.data_transform = {
            'train': transforms.Compose([
                transforms.RandomResizedCrop(resize, scale=(0.5, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ]),
            'val': transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(resize),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])
        }

    def __call__(self, img, phase):
        return self.data_transform[phase](img)

### 코드 6-67 변수에 대한 값 정의

In [ ]:
size = 224
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
batch_size = 32

### 코드 6-68 훈련과 테스트 데이터셋 불러오기

- 개와 고양이 이미지 경로를 모아 두고, cv2로 읽히지 않는 손상된 파일은 걸러 냄

In [ ]:
cat_directory = r'../chap06/data/dogs-vs-cats/Cat/'
dog_directory = r'../chap06/data/dogs-vs-cats/Dog/'

cat_images_filepaths = sorted([os.path.join(cat_directory, f) for f in os.listdir(cat_directory)])
dog_images_filepaths = sorted([os.path.join(dog_directory, f) for f in os.listdir(dog_directory)])
images_filepaths = [*cat_images_filepaths, *dog_images_filepaths]
correct_images_filepaths = [i for i in images_filepaths if cv2.imread(i) is not None]

### 코드 6-69 데이터셋을 훈련, 검증, 테스트 용도로 분리

In [ ]:
random.seed(42)
random.shuffle(correct_images_filepaths)
train_images_filepaths = correct_images_filepaths[:400]
val_images_filepaths = correct_images_filepaths[400:-10]
test_images_filepaths = correct_images_filepaths[-10:]
print(len(train_images_filepaths), len(val_images_filepaths),
      len(test_images_filepaths))

### 코드 6-70 이미지에 대한 레이블 구분

- 파일 이름이 dog면 레이블 1, cat이면 레이블 0을 붙임

In [ ]:
class DogvsCatDataset(Dataset):
    def __init__(self, file_list, transform=None, phase='train'):
        self.file_list = file_list
        self.transform = transform
        self.phase = phase

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path)
        img_transformed = self.transform(img, self.phase)

        label = img_path.split('/')[-1].split('.')[0]
        if label == 'dog':
            label = 1
        elif label == 'cat':
            label = 0
        return img_transformed, label

### 코드 6-71 이미지 데이터셋 정의

In [ ]:
train_dataset = DogvsCatDataset(train_images_filepaths, transform=ImageTransform(size, mean, std), phase='train')
val_dataset = DogvsCatDataset(val_images_filepaths, transform=ImageTransform(size, mean, std), phase='val')

index = 0
print(train_dataset.__getitem__(index)[0].size())
print(train_dataset.__getitem__(index)[1])

### 코드 6-72 데이터셋의 데이터를 메모리로 불러오기

In [ ]:
train_iterator = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_iterator = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
dataloader_dict = {'train': train_iterator, 'val': valid_iterator}

batch_iterator = iter(train_iterator)
inputs, label = next(batch_iterator)
print(inputs.size())
print(label)

### 코드 6-73 기본 블록 정의

- 기본 블록은 ResNet18, ResNet34에서 쓰이고 3x3 합성곱 두 개로 구성됨
- `i = x`로 입력을 따로 저장해 두었다가 마지막에 `x += i`로 더해 줌. 이 부분이 아이덴티티 매핑임
- 입력과 출력의 크기가 다를 때는 다운샘플을 적용해 형태를 맞춤

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        if downsample:
            conv = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                             stride=stride, bias=False)
            bn = nn.BatchNorm2d(out_channels)
            downsample = nn.Sequential(conv, bn)
        else:
            downsample = None
        self.downsample = downsample

    def forward(self, x):
        i = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)

        if self.downsample is not None:
            i = self.downsample(i)

        x += i
        x = self.relu(x)

        return x

### 코드 6-74 병목 블록 정의

- 병목 블록은 ResNet50, ResNet101, ResNet152에서 쓰이고 1x1, 3x3, 1x1 합성곱으로 구성됨
- expansion이 4라서 마지막 1x1 합성곱의 출력 채널이 네 배가 됨

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                               stride=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, self.expansion * out_channels,
                               kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion * out_channels)
        self.relu = nn.ReLU(inplace=True)

        if downsample:
            conv = nn.Conv2d(in_channels, self.expansion * out_channels,
                             kernel_size=1, stride=stride, bias=False)
            bn = nn.BatchNorm2d(self.expansion * out_channels)
            downsample = nn.Sequential(conv, bn)
        else:
            downsample = None
        self.downsample = downsample

    def forward(self, x):
        i = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)

        if self.downsample is not None:
            i = self.downsample(i)

        x += i
        x = self.relu(x)

        return x

### 코드 6-75 ResNet 모델 네트워크

- config로 넘겨받은 block, n_blocks, channels를 가지고 계층을 쌓음
- 7x7 합성곱과 최대 풀링을 지나면 224가 56이 되고, layer1~layer4를 지나며 56 → 28 → 14 → 7로 줄어듦
- zero_init_residual은 각 레지듀얼 분기의 마지막 배치 정규화를 0으로 초기화하는 옵션임. 논문에서 성능이 0.2~0.3% 정도 올랐다고 함

In [ ]:
class ResNet(nn.Module):
    def __init__(self, config, output_dim, zero_init_residual=False):
        super().__init__()

        block, n_blocks, channels = config
        self.in_channels = channels[0]
        assert len(n_blocks) == len(channels) == 4

        self.conv1 = nn.Conv2d(3, self.in_channels, kernel_size=7, stride=2,
                               padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self.get_resnet_layer(block, n_blocks[0], channels[0])
        self.layer2 = self.get_resnet_layer(block, n_blocks[1], channels[1], stride=2)
        self.layer3 = self.get_resnet_layer(block, n_blocks[2], channels[2], stride=2)
        self.layer4 = self.get_resnet_layer(block, n_blocks[3], channels[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(self.in_channels, output_dim)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def get_resnet_layer(self, block, n_blocks, channels, stride=1):
        layers = []
        if self.in_channels != block.expansion * channels:
            downsample = True
        else:
            downsample = False

        layers.append(block(self.in_channels, channels, stride, downsample))

        for i in range(1, n_blocks):
            layers.append(block(block.expansion * channels, channels))

        self.in_channels = block.expansion * channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        h = x.view(x.shape[0], -1)
        x = self.fc(h)
        return x, h

### 코드 6-76 ResNetConfig 정의

In [ ]:
ResNetConfig = namedtuple('ResNetConfig', ['block', 'n_blocks', 'channels'])

### 코드 6-77 기본 블록을 사용하여 ResNetConfig 정의

In [ ]:
resnet18_config = ResNetConfig(block=BasicBlock,
                               n_blocks=[2,2,2,2],
                               channels=[64,128,256,512])

resnet34_config = ResNetConfig(block=BasicBlock,
                               n_blocks=[3,4,6,3],
                               channels=[64,128,256,512])

### 코드 6-78 병목 블록을 사용하여 ResNetConfig 정의

In [ ]:
resnet50_config = ResNetConfig(block=Bottleneck,
                               n_blocks=[3,4,6,3],
                               channels=[64,128,256,512])

resnet101_config = ResNetConfig(block=Bottleneck,
                                n_blocks=[3,4,23,3],
                                channels=[64,128,256,512])

resnet152_config = ResNetConfig(block=Bottleneck,
                                n_blocks=[3,8,36,3],
                                channels=[64,128,256,512])

### 코드 6-79 사전 훈련된 ResNet 모델 사용

In [ ]:
pretrained_model = models.resnet50(pretrained=True)

### 코드 6-80 사전 훈련된 ResNet 네트워크 확인

In [ ]:
print(pretrained_model)

### 코드 6-81 ResNet50 Config를 사용한 ResNet 모델 사용

- 개와 고양이 두 개의 클래스를 쓰므로 OUTPUT_DIM은 2로 둠
- 직접 만든 네트워크를 출력해 보면 사전 훈련된 ResNet50과 구조가 다르지 않음을 확인할 수 있음

In [ ]:
OUTPUT_DIM = 2
model = ResNet(resnet50_config, OUTPUT_DIM)
print(model)

### 코드 6-82 옵티마이저와 손실 함수 정의

- lr=1e-7은 1 곱하기 10의 -7승을 의미함

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-7)
criterion = nn.CrossEntropyLoss()

model = model.to(device)
criterion = criterion.to(device)

scheduler = None

### 코드 6-83 모델 학습 정확도 측정 함수 정의

- topk는 가장 큰 값의 인덱스를 가져오는 함수임. 네트워크 출력에서 확률이 가장 높은 값의 인덱스를 반환함
- t()는 차원 0과 1을 전치한다는 뜻임
- eq는 두 텐서의 요소를 비교해서 같으면 True, 다르면 False를 반환함
- 클래스가 두 개뿐이라 top-2는 항상 정답을 포함하므로 acc_5는 100%로 나옴

In [ ]:
def calculate_topk_accuracy(y_pred, y, k=2):
    with torch.no_grad():
        batch_size = y.shape[0]
        _, top_pred = y_pred.topk(k, 1)
        top_pred = top_pred.t()
        correct = top_pred.eq(y.view(1, -1).expand_as(top_pred))
        correct_1 = correct[:1].reshape(-1).float().sum(0, keepdim=True)
        correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
        acc_1 = correct_1 / batch_size
        acc_k = correct_k / batch_size
    return acc_1, acc_k

- topk, t(), eq의 동작을 교재 예시로 확인함

In [ ]:
import torch
x = torch.arange(1., 6.)
print(x)
print('-----------------')
print(torch.topk(x, 3))

In [ ]:
x = torch.randn(3)
print(x)
print(torch.t(x))
print('------------')
x = torch.randn(2, 3)
print(x)
print(torch.t(x))

In [ ]:
print(torch.eq(torch.tensor([[1, 2], [3, 4]]), torch.tensor([[1, 1], [4, 4]])))

### 코드 6-84 모델 학습 함수 정의

In [ ]:
def train(model, iterator, optimizer, criterion, scheduler, device):
    epoch_loss = 0
    epoch_acc_1 = 0
    epoch_acc_5 = 0

    model.train()
    for (x, y) in iterator:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred[0], y)

        acc_1, acc_5 = calculate_topk_accuracy(y_pred[0], y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc_1 += acc_1.item()
        epoch_acc_5 += acc_5.item()

    epoch_loss /= len(iterator)
    epoch_acc_1 /= len(iterator)
    epoch_acc_5 /= len(iterator)
    return epoch_loss, epoch_acc_1, epoch_acc_5

### 코드 6-85 모델 평가 함수 정의

In [ ]:
def evaluate(model, iterator, criterion, device):
    epoch_loss = 0
    epoch_acc_1 = 0
    epoch_acc_5 = 0

    model.eval()
    with torch.no_grad():
        for (x, y) in iterator:
            x = x.to(device)
            y = y.to(device)
            y_pred = model(x)
            loss = criterion(y_pred[0], y)

            acc_1, acc_5 = calculate_topk_accuracy(y_pred[0], y)
            epoch_loss += loss.item()
            epoch_acc_1 += acc_1.item()
            epoch_acc_5 += acc_5.item()

    epoch_loss /= len(iterator)
    epoch_acc_1 /= len(iterator)
    epoch_acc_5 /= len(iterator)
    return epoch_loss, epoch_acc_1, epoch_acc_5

### 코드 6-86 모델 학습 시간 측정 함수 정의

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

### 코드 6-87 모델 학습

- 검증 손실이 가장 낮을 때의 가중치를 저장함

In [ ]:
best_valid_loss = float('inf')
EPOCHS = 10

for epoch in range(EPOCHS):
    start_time = time.monotonic()

    train_loss, train_acc_1, train_acc_5 = train(model, train_iterator, optimizer,
                                                 criterion, scheduler, device)
    valid_loss, valid_acc_1, valid_acc_5 = evaluate(model, valid_iterator, criterion,
                                                    device)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), '../chap06/data/ResNet-model.pt')

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc @1: {train_acc_1*100:6.2f}% | ' \
          f'Train Acc @5: {train_acc_5*100:6.2f}%')
    print(f'\tValid Loss: {valid_loss:.3f} | Valid Acc @1: {valid_acc_1*100:6.2f}% | ' \
          f'Valid Acc @5: {valid_acc_5*100:6.2f}%')

- 학습률이 1e-7로 매우 작고 이미지 수도 400개뿐이라 오차와 정확도 모두 좋게 나오지는 않음
- 이 예제의 목적은 성능 향상이 아니라 CNN 관련 네트워크의 사용 방법을 익히는 것임

### 코드 6-88 테스트 데이터셋을 이용한 모델 예측

- 예측 결과를 ResNet.csv로 저장함
- label이 0.5보다 크면 개, 0.5보다 작으면 고양이를 의미함

In [ ]:
import pandas as pd
id_list = []
pred_list = []
_id = 0
with torch.no_grad():
    for test_path in test_images_filepaths:
        img = Image.open(test_path)
        _id = test_path.split('/')[-1].split('.')[1]
        transform = ImageTransform(size, mean, std)
        img = transform(img, phase='val')
        img = img.unsqueeze(0)
        img = img.to(device)

        model.eval()
        outputs = model(img)
        preds = F.softmax(outputs[0], dim=1)[:, 1].tolist()
        id_list.append(_id)
        pred_list.append(preds[0])

res = pd.DataFrame({
    'id': id_list,
    'label': pred_list
})

res.sort_values(by='id', inplace=True)
res.reset_index(drop=True, inplace=True)

res.to_csv('../chap06/data/ResNet.csv', index=False)
res.head(10)

### 코드 6-89 모델 예측에 대한 결과 출력

In [ ]:
class_ = classes = {0:'cat', 1:'dog'}

def display_image_grid(images_filepaths, predicted_labels=(), cols=5):
    rows = len(images_filepaths) // cols
    figure, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(12, 6))
    for i, image_filepath in enumerate(images_filepaths):
        image = cv2.imread(image_filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        a = random.choice(res['id'].values)
        label = res.loc[res['id'] == a, 'label'].values[0]

        if label > 0.5:
            label = 1
        else:
            label = 0
        ax.ravel()[i].imshow(image)
        ax.ravel()[i].set_title(class_[label])
        ax.ravel()[i].set_axis_off()
    plt.tight_layout()
    plt.show()

display_image_grid(test_images_filepaths)

## 정리

- 기본 블록과 병목 블록, 아이덴티티 매핑, 다운샘플까지 직접 정의해서 ResNet50을 만들어 봄
- 직접 만든 네트워크와 `models.resnet50(pretrained=True)`로 불러온 네트워크의 구조가 같다는 것을 출력으로 확인함
- 앞으로 ResNet이 필요하면 사전 훈련된 모델을 한 줄로 불러 쓰면 되지만, 구조를 알고 쓰는 것과 모르고 쓰는 것은 차이가 큼
- 클래스가 두 개라 Acc @5(실제로는 top-2)가 항상 100%로 나오므로 성능은 Acc @1로 봐야 함
- 학습률 1e-7, 데이터 400장, 10 에포크 조건에서는 손실이 거의 줄지 않음. 성능을 올리려면 데이터 수와 학습률부터 손봐야 함